# Worksheet 2: Gradio Integration Workshop

**Name:** Nazima Akhter Neha **Date:** 03/05/2026

## Learning Objectives
By the end of this worksheet, you will:

- Understand how Gradio components are also objects
- Connect your Transaction and Manager classes to a user interface
- Build a working prototype of your Smart Finance App
- See how different types of objects collaborate in a real application

In [22]:
---

## Setup: Install and Import

```python
# Install required packages (run this first)
!pip install gradio pandas

# Import what we need
import gradio as gr
import pandas as pd
from datetime import datetime, date
import json
```

---

## Part 1: Gradio Objects Discovery

### Task 1.1: Understanding Gradio as Objects

Before we integrate, let's explore how Gradio itself uses object-oriented design. 

Ask AI about this:

> "I'm building a user interface for a business app. When I create components like text input boxes, buttons, and dropdowns, I'm making interactive elements that users can click and type in. How is this similar to the way businesses create standardized forms - like how every customer order form has the same structure but holds different information for each customer?"

**AI's explanation:** Gradio components are like objects because each component has its own data and behaviour. For example, a textbox stores a label, a placeholder, and user input. A button waits for a click and can run a function. This is similar to business objects, where each object stores information and performs actions.

### Task 1.2: Simple Gradio Object Exploration

Let's see Gradio objects in action:

```python
# Create some basic Gradio objects and see their properties
textbox = gr.Textbox(label="Transaction Description", placeholder="Enter description...")
number_input = gr.Number(label="Amount", value=0.0)
dropdown = gr.Dropdown(label="Category", choices=["Food", "Transport", "Entertainment", "Bills", "Income"])

# Print information about these objects
print(f"Textbox type: {type(textbox)}")
print(f"Number input type: {type(number_input)}")
print(f"Dropdown type: {type(dropdown)}")
```

**What did you discover about Gradio components?**
I discovered that Gradio components are Python objects. Each component has a type, such as Textbox, Number, or Dropdown, and each object stores settings like labels, values, and choices.
---

## Part 2: Building Your Transaction Class (Review & Enhance)

### Task 2.1: Core Transaction Class

Let's start with the Transaction class you developed in Worksheet 1. If you need help, ask AI:

> "I need to create a template for tracking financial transactions in my app. Each transaction should remember its description, amount (negative for expenses, positive for income), category, and date. The template should also be able to tell me if it's an expense and display itself nicely. How would you design this?"

```python
# Your Transaction class (from Worksheet 1 or AI-generated):

class Transaction:
    def __init__(self, description, amount, category, date=None):
       self.description = description
       self.amount = float(amount)
       self.category = category
       self.date = date 
       
    
    def is_expense(self):
        return self.amount < 0
    
    def __str__(self):
       if self.is_expensive():
           transaction_type = "Expense"
       else:
           transaction_type = "Income"
       return f"{self.description} | ${self.amount:.2f} | {self.category} | {transaction_type}"
        
# Test your Transaction class:
test_transaction = Transaction("Coffee", -4.50, "Food")
print(test_transaction)
print(f"Is expense: {test_transaction.is_expense()}")
```

---

## Part 3: Simple Finance Manager System

### Task 3.1: Building the Manager Class

Now create a system to manage multiple transactions:

```python
class SimpleFinanceManager:
    def __init__(self):
        self.transactions = []
    
    def add_transaction(self, transaction):
        self.transactions.append(transaction)
    
    def get_total_expenses(self):
        return sum(t.amount for t in self.transactions if t.amount < 0)
        
    def get_spending_by_category(self, category):
       return sum(t.amount for t in self.transactions if t.category == category)
    
    def get_recent_transactions(self, count=5):
       return self.transactions[-count:]
        
    def get_summary(self):
        total_expenses = self.get_total_expenses()
        total_income = sum(t.amount for t in self.transactions if t.amount >0)
        balance = total_income + total_expenses

        return f"Total Income: ${total_income:.2f}\nTotal Expenses: $(total_expenses:.2f}\nBalance: ${balance:.2f}\nTransactions: {len(self.transactions)}"

# Test your manager:
manager = SimpleFinanceManager()

manager.add_transaction(Transaction("Coffee", -4.50, "Food"))
manager.add_transaction(Transaction("Salary", 500.00, "Income"))
manager.add_transaction(Transaction("Bus Ticket", -3.20, "Transport"))

print(manager.get.summary())
print(manager.get_spending_by_category("Food"))

=[[]# Add some test transactions and verify it works
```

---

## Part 4: Connecting Objects to Gradio Interface

### Task 4.1: The Connection Function

This is where the magic happens - your objects work with Gradio objects. Ask AI for help:

> "I have a transaction template and a transaction manager system. I want to create a web form where users can enter transaction details, and when they click 'Add', it should create a new transaction and add it to my manager, then show a confirmation message. How do I connect my business logic to a user interface?"

```python
# Create a global manager instance
app_manager = SimpleFinanceManager()

def add_transaction_via_gradio(description, amount, category):
    """This function connects Gradio inputs to your objects"""
    try:
        transaction = Transaction(description, amount, category)
        app_manager.add_transaction(transaction)
        return f"Added transaction: {transaction}"    
    except Exception as e:
        return f"Error: {str(e)}"

def get_spending_summary():
    return app_manager.get_summary()
# Test the function manually first:
result = add_transaction_via_gradio("Test Coffee", -4.50, "Food")
print(result)
```

### Task 4.2: Building the Interface

Now create the actual Gradio interface:

```python
# Create the Gradio interface
with gr.Blocks(title="Smart Finance App Prototype") as demo:
    
    gr.Markdown("# 💰 Smart Finance App")
    gr.Markdown("Add transactions and see your spending patterns!")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("## Add New Transaction")
            
            # Create Gradio input objects
            desc_input = gr.Textbox(label="Description", placeholder="e.g., Starbucks Coffee")
            amount_input = gr.Number(label="Amount", value=0.0, info="Negative for expenses, positive for income")
            category_input = gr.Dropdown(label="Category", 
                                       choices=["Food", "Transport", "Entertainment", "Bills", "Income", "Other"])
            
            add_button = gr.Button("Add Transaction", variant="primary")
            
        with gr.Column():
            gr.Markdown("## Summary")
            summary_display = gr.Textbox(label="Current Summary", interactive=False)
            refresh_button = gr.Button("Refresh Summary")
    
    # Status area
    status_output = gr.Textbox(label="Status", interactive=False)
    
    # Connect the objects: Button objects call functions that use your custom objects
    add_button.click(
        fn=add_transaction_via_gradio,
        inputs=[desc_input, amount_input, category_input],
        outputs=status_output
    )
    
    refresh_button.click(
        fn=get_spending_summary,
        outputs=summary_display
    )

# Launch the app
demo.launch(debug=True)
```

---

## Part 5: Testing Object Collaboration

### Task 5.1: Integration Testing

Test your app by adding several transactions and observing how the objects work together:

**Add these test transactions:**
1. Coffee Shop: -$4.50, Food
2. Bus Ticket: -$3.20, Transport  
3. Salary: +$500.00, Income
4. Groceries: -$67.80, Food

**Document what happens:**

**1. Object Creation:** When you click "Add Transaction", trace what objects get created:

**2. Object Interaction:** How do the Gradio objects pass data to your Transaction objects?

**3. System Updates:** How does the SimpleFinanceManager coordinate everything?

---

## Part 6: Advanced Integration Challenge

### Task 6.1: CSV Loading Feature

Let's add the ability to load transactions from CSV files. Ask AI:

> "I want to add a file upload feature to my finance app so users can load their bank transaction data from CSV files. The CSV has columns for description, amount, category, and date. How do I take this spreadsheet data and integrate it with my existing transaction management system?"

```python
def load_transactions_from_csv(csv_file):
    """Load transactions from uploaded CSV file"""
    if csv_file is None:
        return "No file uploaded"
    
    try:
        df = pd.read_csv(csv_file.name)

        for index, row in df.iterrows():
            transaction = Transaction(
                row["description"],
                row["amount"],
                row["category"],
                row.get("date", None)
            )
            app_manager.add_transaction(transaction)

         return f"Loaded {len(df)} transactions from CSV."
            
    except Exception as e:
        return f"Error loading CSV: {str(e)}"

# Add this to your Gradio interface (create a new version):
# Include a gr.File() component and connect it to your function
```

### Task 6.2: Enhanced Interface

Create an enhanced version of your interface that includes CSV loading:

```python
# Enhanced interface with CSV loading
with gr.Blocks(title="Smart Finance App v2") as enhanced_demo:
    
    gr.Markdown("# 💰 Smart Finance App v2")
    
    with gr.Tab("Add Transactions"):
        # Your manual transaction entry interface
        pass
    
    with gr.Tab("Load from CSV"):
        # Your CSV loading interface
        pass
    
    with gr.Tab("Analysis"):
        # Your summary and analysis interface
        pass

# Test the enhanced version
enhanced_demo.launch(debug=True)
```

---

## Part 7: Problem-Solving Analysis

### Task 7.1: Object Collaboration Mapping

Draw or describe how the different objects in your system work together:

**Gradio Objects:** (What Gradio objects did you use?)
I used Textbox, Number, Dropdown, Button, Markdown, Row, Column, Blocks, File, and Tab components.

**Your Custom Objects:** (Transaction, SimpleFinanceManager)
I used a Transaction object to represent one financial transaction and a SimpleFinanceManager object to store and manage many transactions.
**Data Flow:** (How does information move between objects?)
The user enters data into Gradio components. When the button is clicked, Gradio sends the input data to a Python function. The function creates a Transaction object and adds it to the SimpleFinanceManager. The manager then updates the summary and sends the result back to the Gradio interface.
### Task 7.2: Business Problem Solved

**1. Integration Problem:** How did your solution handle both manual entry AND CSV loading using the same objects?
The solution handles both manual entry and CSV loading by converting both types of input into the same Transaction objects. This makes the system easier to manage because all transactions are stored in the same manager.

**2. User Experience:** How do objects make the interface more reliable and user-friendly?
Objects make the interface more reliable because each part has a clear job. Gradio handles the user interface, Transaction handles individual data, and SimpleFinanceManager handles calculations and summaries.

**3. Scalability:** How would your object-oriented design handle more features (budgets, categories, reports)?
The design can handle more features by adding new methods or classes. For example, budget tracking, reports, and category analysis can be added without rewriting the whole system.

---

## Part 8: AI-Assisted Enhancement

### Task 8.1: Feature Expansion

Ask AI to help you add one more feature to your finance app:

> "I want to add [CHOOSE: budget tracking / expense categories analysis / monthly reports / spending alerts] to my finance app. How would I enhance my existing transaction and manager systems to support this business feature while keeping everything organized and easy to maintain?"

```python
# AI's suggested enhancement:
A useful new feature would be spending alerts. The finance app can warn the user when an expense is very large, such as over $50. This helps users notice unusual spending and manage their money better.

```

### Task 8.2: Implementation and Testing

Implement the AI's suggestion and test it:

```python
# Your enhanced classes:
class EnhancedFinanceManager(SimpleFinanceManager):
    def get_spending_alerts(self):
        alerts = []

        for transaction in self.transactions:
            if transaction.amount < -50:
                alerts.append(f"Large expense alert: {transaction.description} cost $(abs(transaction.amount):.2f}")

         if len(alerts) == 0: 
             return "No spending alerts."

        return "\n".join(alerts)


enhanced_manager = EnhancedFinanceManager()
enhanced_manager.add_transaction(Transaction("Coffee", -4.50, "Food"))
enhanced_manager.add_transaction(Transaction("Groceries", -67.80, "Food"))
enhanced_manager.add_transaction(Transaction("Salary", 500.00,"Income"))

print(enhanced_manager.get_spending_alerts())

```

```python
# Test the new feature:

```

---

## Reflection Questions

### Task 9.1: OOP Problem-Solving Insights

**1. Object Collaboration:** How did using multiple types of objects (Gradio objects + your custom objects) solve complex problems?
Using multiple objects helped solve the problem because each object had a specific responsibility. Gradio objects collected user input, Transaction objects stored financial data, and the manager object organized and analysed all transactions.
**2. Separation of Concerns:** How did keeping business logic (Transaction, Manager) separate from interface logic (Gradio) help?
Keeping business logic separate from interface logic made the program easier to understand and maintain. The finance calculations are inside the manager class, while Gradio only handles the user interface.
**3. Real-World Application:** How is this pattern used in apps you use daily?
This pattern is used in banking apps, shopping apps, and food delivery apps. For example, a banking app uses transaction objects, account objects, and interface objects to show balances, payments, and spending history.
### Task 9.2: AI Development Partnership

**1. AI as Design Partner:** How did AI help you explore solutions you wouldn't have thought of?
AI helped me explore solutions by suggesting how to connect my Transaction and SimpleFinanceManager classes to a Gradio interface. It also helped me understand how buttons, textboxes, dropdowns, and functions work together in an application.

**2. Human Oversight:** Where did you need to guide, correct, or enhance AI suggestions?
I needed to guide and correct the AI by checking the code, making sure expenses used negative numbers, testing the functions, and fixing missing or incomplete parts. Human checking was important because AI suggestions are not always perfect.

**3. Problem-Solving Process:** How did AI change your approach to building software?
AI changed my approach by helping me break the software into smaller parts. Instead of trying to build the whole app at once, I worked step by step: create objects, test them, connect them to Gradio, then improve the app.
---

## Extension Challenges (Optional)

### Challenge 1: Smart Chatbot Integration
Add a simple chatbot that can answer questions about spending using your objects.

### Challenge 2: Data Visualization  
Add charts to visualize spending patterns using your transaction data.

### Challenge 3: Export Functionality
Add the ability to export transaction data back to CSV format.

---

## Key Takeaways

You've now built a working finance application that demonstrates:

- **Object Collaboration:** How different types of objects work together
- **Separation of Concerns:** UI objects vs business logic objects
- **Scalable Design:** How OOP makes adding features easier  
- **Real-World Integration:** How to connect file data, manual input, and user interfaces
- **AI-Assisted Development:** How to effectively partner with AI in building applications

Most importantly: You've seen how object-oriented programming solves real business problems by organising code the same way businesses organise their operations.

SyntaxError: unterminated string literal (detected at line 22) (2376638351.py, line 22)

In [ ]:
import pandas as pd

# Function to export transactions to CSV file
def export_transaction_csv(manager, filename="transactions.csv"):
    # Creating a list of transactions to add to DataFrame
    data = []
    for transaction in manager.transactions: 
        data.append({
            "Description": transaction.description,
            "Amount": transaction.amount,
            "Category": transaction.category,
            "Date": transaction.date
        })

    # Convert the data to a DataFrame 
    df = pd.DataFrame(data)

    # Exporting the data to a CSV file
    df.to_csv(filename,index=False)

    return f"Data has been exported to {filename}"

# Example use: 
export_transactions_to_csv(manager)
        